# Company Insights 360 — Data Analysis & Cleaning Notebook

This notebook demonstrates beginner-friendly data analysis using **Python** and **Pandas** for the Company Insights 360 project.
The workflow includes data loading, missing value checks, data type conversions, exploratory aggregations, and visual chart generation.

## 1. Data Loading & Initial Inspection

In this section, we import required libraries (`pandas`, `matplotlib`) and load the three core datasets: `departments.csv`, `employees.csv`, and `sales.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

# Configure chart styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully!")

In [ ]:
# Load CSV datasets from the data/ directory
departments = pd.read_csv("../data/departments.csv")
employees = pd.read_csv("../data/employees.csv")
sales = pd.read_csv("../data/sales.csv")

print(f"Departments shape: {departments.shape}")
print(f"Employees shape: {employees.shape}")
print(f"Sales shape: {sales.shape}")

### Inspecting `departments` Table

In [ ]:
print("--- DEPARTMENTS INFO ---")
departments.info()
display(departments.head())

### Inspecting `employees` Table

In [ ]:
print("--- EMPLOYEES INFO ---")
employees.info()
display(employees.head())
display(employees.describe())

### Inspecting `sales` Table

In [ ]:
print("--- SALES INFO ---")
sales.info()
display(sales.head())
display(sales.describe())

## 2. Data Cleaning & Validation

Here we perform data quality audits:
1. Checking missing values (`isnull().sum()`)
2. Checking duplicate records (`duplicated().sum()`)
3. Converting date strings to `datetime` objects
4. Verifying foreign key integrity between tables

In [ ]:
# 2.1 Check missing values
print("Missing values in Departments:")
print(departments.isnull().sum())
print("\nMissing values in Employees:")
print(employees.isnull().sum())
print("\nMissing values in Sales:")
print(sales.isnull().sum())

In [ ]:
# 2.2 Check duplicate rows
print("Duplicate rows in Departments:", departments.duplicated().sum())
print("Duplicate rows in Employees:", employees.duplicated().sum())
print("Duplicate rows in Sales:", sales.duplicated().sum())

In [ ]:
# 2.3 Convert date fields to datetime
employees['HireDate'] = pd.to_datetime(employees['HireDate'])
sales['Date'] = pd.to_datetime(sales['Date'])

print("HireDate dtype:", employees['HireDate'].dtype)
print("Sales Date dtype:", sales['Date'].dtype)

In [ ]:
# 2.4 Foreign key validity checks
unmatched_depts = set(employees['Department']) - set(departments['Department'])
unmatched_emps = set(sales['EmployeeID']) - set(employees['EmployeeID'])

print("Unmatched Departments in Employees:", unmatched_depts)
print("Unmatched EmployeeIDs in Sales:", unmatched_emps)

## 3. Exploratory Data Analysis & Aggregations

Now we calculate core business metrics matching our SQL analysis:

### 3.1 Department Headcount & Salary Analysis

In [ ]:
# Headcount by Department
dept_summary = employees.groupby('Department').agg(
    Total_Employees=('EmployeeID', 'count'),
    Avg_Salary=('Salary', 'mean'),
    Avg_Performance=('PerformanceScore', 'mean')
).reset_index()

dept_summary['Avg_Salary'] = dept_summary['Avg_Salary'].round(2)
dept_summary['Avg_Performance'] = dept_summary['Avg_Performance'].round(2)
display(dept_summary.sort_values(by='Total_Employees', ascending=False))

### 3.2 Sales & Profitability by Region

In [ ]:
region_summary = sales.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

region_summary['Profit_Margin_%'] = ((region_summary['Total_Profit'] / region_summary['Total_Sales']) * 100).round(2)
region_summary['Total_Sales'] = region_summary['Total_Sales'].round(2)
region_summary['Total_Profit'] = region_summary['Total_Profit'].round(2)
display(region_summary.sort_values(by='Total_Sales', ascending=False))

### 3.3 Sales Performance by Product Category

In [ ]:
category_summary = sales.groupby('Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

category_summary['Total_Sales'] = category_summary['Total_Sales'].round(2)
category_summary['Total_Profit'] = category_summary['Total_Profit'].round(2)
display(category_summary.sort_values(by='Total_Sales', ascending=False))

### 3.4 Yearly Revenue Trend

In [ ]:
sales['Year'] = sales['Date'].dt.year
yearly_trend = sales.groupby('Year').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

yearly_trend['Total_Sales'] = yearly_trend['Total_Sales'].round(2)
yearly_trend['Total_Profit'] = yearly_trend['Total_Profit'].round(2)
display(yearly_trend)

### 3.5 Top 5 Employees by Sales Revenue

In [ ]:
emp_sales = sales.groupby('EmployeeID')['Sales'].sum().reset_index()
top_sellers = pd.merge(emp_sales, employees[['EmployeeID', 'Name', 'Department']], on='EmployeeID')
top_sellers['Sales'] = top_sellers['Sales'].round(2)
display(top_sellers.sort_values(by='Sales', ascending=False).head(5))

## 4. Simple Data Visualizations

We create clean bar and line charts to present key findings:

In [ ]:
# Chart 1: Revenue by Region
plt.figure(figsize=(8, 4))
plt.bar(region_summary['Region'], region_summary['Total_Sales'], color='#1f77b4')
plt.title("Total Revenue by Region", fontsize=14, fontweight="bold")
plt.xlabel("Region")
plt.ylabel("Revenue (INR)")
plt.tight_layout()
plt.savefig("chart_revenue_by_region.png")
plt.close()

In [ ]:
# Chart 2: Average Salary by Department
plt.figure(figsize=(8, 4))
plt.bar(dept_summary['Department'], dept_summary['Avg_Salary'], color='#2ca02c')
plt.title("Average Salary by Department", fontsize=14, fontweight="bold")
plt.xlabel("Department")
plt.ylabel("Avg Salary (INR)")
plt.tight_layout()
plt.savefig("chart_salary_by_dept.png")
plt.close()

In [ ]:
# Chart 3: Yearly Revenue Trend
plt.figure(figsize=(8, 4))
plt.plot(yearly_trend['Year'], yearly_trend['Total_Sales'], marker='o', color='#ff7f0e', linewidth=2)
plt.title("Yearly Revenue Trend (2020 - 2024)", fontsize=14, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Revenue (INR)")
plt.xticks(yearly_trend['Year'])
plt.tight_layout()
plt.savefig("chart_yearly_trend.png")
plt.close()

## 5. Key Business Insights Summary

1. **Departmental Distribution**: Marketing has the highest headcount (23 employees), while Finance and HR have the lowest (17 employees each).
2. **Compensation**: Finance offers the highest average salary (84,917.65 INR), followed by IT (84,159.09 INR).
3. **Regional Revenue**: South leads in total revenue (1,372,223.31 INR), whereas West achieves the highest profit margin percentage (15.86%).
4. **Category Performance**: Software is the top-performing product category, generating 1,211,686.62 INR in total revenue.
5. **Growth Trend**: Annual revenue peaked in 2023 at 1,143,544.39 INR.